# Mini-projet Data Mining — Détection automatique des crises d'absence à partir de signaux EEG

**Binôme :** _(à compléter)_
**Format des données :** chaque patient a deux fichiers exportés depuis MATLAB (timetable) vers CSV :
- `<id>_0000d__output_d.csv` : le signal EEG brut, au format **large** (1 ligne = 1 seconde, colonnes `<canal>_<1..256>` pour les 256 échantillons de cette seconde, 23 canaux : 19 EEG + EMG1/EMG2/SLI/ECG).
- `<id>_0000h__output_h.csv` : les annotations/événements de la séance (colonnes `Onset`, `Annotations`, `Duration`).

Les crises d'absence elles-mêmes (début/fin par patient) sont dans le fichier séparé `Crises_absences.xlsx` (Partie II).

In [ ]:
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 10)
plt.rcParams['figure.figsize'] = (12, 4)

## Configuration

Adapter `DATA_DIR` et `PATIENT_ID` selon le patient à analyser.

In [ ]:
DATA_DIR = "./data"          # dossier contenant les CSV convertis
PATIENT_ID = "190304A_E_0000" # identifiant du patient/enregistrement

signal_path = f"{DATA_DIR}/{PATIENT_ID}d__output_d.csv"
annotations_path = f"{DATA_DIR}/{PATIENT_ID}h__output_h.csv"

## Partie I — Compréhension et exploration des données

### Question 1

Le fichier signal est fourni dans un format **large** : chaque ligne correspond à **1 seconde**
d'enregistrement, et contient les 256 échantillons de cette seconde pour chacun des 23 canaux
(colonnes `<canal>_1` à `<canal>_256`). Avant de répondre aux questions, on le transforme donc en
un format **long** classique : une colonne `Time` (en secondes) + une colonne par canal, avec une
ligne par échantillon — c'est la structure `Time, EEG canal 1, EEG canal 2, ...` attendue par le sujet.

In [ ]:
def load_eeg_wide(path):
    """Charge le CSV large exporte depuis MATLAB (1 ligne = 1 seconde de signal)."""
    return pd.read_csv(path)


def wide_to_long(df_wide):
    """
    Convertit le format large (1 ligne = 1s, colonnes <canal>_<echantillon>) en format long :
    une colonne 'Time' (secondes) + une colonne par canal (une ligne par echantillon).
    Retourne aussi la frequence d'echantillonnage detectee et la liste des canaux.
    """
    sample_cols = [c for c in df_wide.columns if c != 'Record Time']
    channels = sorted(
        set(re.sub(r'_\d+$', '', c) for c in sample_cols),
        key=lambda ch: sample_cols.index(f"{ch}_1")
    )

    # nombre d'echantillons par ligne (= par seconde) pour le premier canal
    samples_per_row = sum(1 for c in sample_cols if c.startswith(channels[0] + "_"))

    # frequence d'echantillonnage = nb d'echantillons par ligne / duree d'une ligne (Record Time)
    dt_row = df_wide['Record Time'].diff().dropna().median()
    fs = samples_per_row / dt_row

    n_rows = len(df_wide)
    n_samples = n_rows * samples_per_row
    time = np.arange(n_samples) / fs

    data = {}
    for ch in channels:
        cols = [f"{ch}_{i}" for i in range(1, samples_per_row + 1)]
        data[ch] = df_wide[cols].to_numpy().reshape(-1)

    df_long = pd.DataFrame(data)
    df_long.insert(0, 'Time', time)
    return df_long, channels, fs


df_wide = load_eeg_wide(signal_path)
df, channels, fs = wide_to_long(df_wide)

print(f"Fichier charge : {signal_path}")
print(f"Nombre de lignes (echantillons)      : {df.shape[0]}")
print(f"Nombre de variables (colonnes)       : {df.shape[1]}  (dont 1 colonne Time + {len(channels)} canaux)")
print(f"Frequence d'echantillonnage detectee  : {fs:.2f} Hz")
print(f"Duree totale de l'enregistrement      : {df['Time'].iloc[-1] - df['Time'].iloc[0]:.2f} s "
      f"(~{(df['Time'].iloc[-1] - df['Time'].iloc[0])/60:.1f} min)")
print(f"Nombre de valeurs manquantes (total)  : {df.isna().sum().sum()}")

FileNotFoundError: [Errno 2] No such file or directory: './data/190304A_E_0000d__output_d.csv'

**Aperçu des premières lignes du fichier :**

In [ ]:
df.head(10)

NameError: name 'df' is not defined

**Visualisation rapide (premières secondes, un canal) :**

In [ ]:
preview_seconds = 5
n_preview = int(preview_seconds * fs)

plt.plot(df['Time'][:n_preview], df[channels[0]][:n_preview])
plt.xlabel("Temps (s)")
plt.ylabel("Amplitude EEG (µV)")
plt.title(f"Aperçu du signal — canal {channels[0]} — {preview_seconds} premières secondes")
plt.tight_layout()
plt.show()

NameError: name 'fs' is not defined

### Question 2

Représentation du signal EEG en fonction du temps (axe horizontal : temps en secondes,
axe vertical : amplitude EEG en µV), pour un canal donné.

In [ ]:
channel_to_plot = channels[0]  # changer ici pour visualiser un autre canal

plt.figure(figsize=(14, 4))
plt.plot(df['Time'], df[channel_to_plot], linewidth=0.5)
plt.xlabel("Temps (s)")
plt.ylabel("Amplitude EEG (µV)")
plt.title(f"Signal EEG — canal {channel_to_plot} — patient {PATIENT_ID}")
plt.tight_layout()
plt.show()

**Question :** Que remarquez-vous visuellement dans les périodes correspondant aux crises ?

_(à compléter après avoir croisé ce signal avec les instants de crise de la Partie II —
on s'attend typiquement à des bouffées d'ondes plus amples et plus régulières pendant les crises d'absence)._

## Partie II — Exploitation des annotations

**Méthodologie retenue :** le fichier `Crises_absences.xlsx` donne le début/fin de chaque crise
en **heures d'horloge absolues** (fractions de jour au format Excel), alors que le signal EEG est
indexé en **secondes relatives** au début de l'enregistrement. Sans l'heure d'horloge exacte du
début de l'enregistrement (perdue à l'export MATLAB → CSV), impossible de convertir directement
les unes vers les autres — **mais la différence entre deux temps du xlsx reste une durée exacte
et fiable** (la conversion de date s'annule dans la soustraction).

On combine donc deux sources complémentaires :

1. **Le fichier d'annotations `_h.csv`** (référentiel temporel du signal, en secondes) :
   - certaines crises y sont bornées explicitement par un triplet `a` (début) → `crise` (pic) → `b` (fin) ;
   - d'autres ne sont marquées que par une annotation isolée contenant `crise` (ex. `crise??`,
     point d'interrogation du clinicien = incertitude), **sans borne `a`/`b` explicite**.
2. **Le fichier `Crises_absences.xlsx`** : sert (a) de vérification croisée du nombre de crises
   (`NombreCE`), et (b) à **combler la borne de fin manquante** des crises isolées, en leur
   attribuant la durée exacte `(FinCEn − DebCEn) × 86400`, appariée dans l'ordre chronologique.

Cette approche a été validée sur 2 patients aux profils différents :
- `190304A_E_0000` : 1 crise, entièrement bornée par `a`/`crise`/`b` ;
- `191113A_D_0000` : 2 crises — une bornée par `a`/`b`, une seconde marquée seulement `crise??`
  (durée complétée via le xlsx). Le nombre total détecté (2) est bien cohérent avec `NombreCE=2`.

In [ ]:
def load_annotations(path):
    """Charge le fichier d'annotations _h.csv et nettoie la colonne Annotations
    (encodage casse + espaces + minuscules) pour faciliter la recherche de marqueurs."""
    ann = pd.read_csv(path)
    ann['Annotations_clean'] = (
        ann['Annotations'].astype(str)
        .str.replace('\ufffd\ufffd', 'e', regex=False)   # corrige l'encodage casse ("d\ufffd\ufffdbut" -> "debut")
        .str.strip()
        .str.lower()
    )
    return ann


def get_xlsx_crisis_durations(xlsx_row):
    """Retourne la liste des durees exactes (s) des crises d'apres les colonnes DebCEn/FinCEn
    du xlsx (gere les variantes de nommage 'DebCE5', 'DebutCE5', 'DebtCE11', ...),
    triees par n (on suppose que les colonnes sont dans l'ordre chronologique)."""
    deb_cols, fin_cols = {}, {}
    for col in xlsx_row.index:
        m = re.match(r'^Deb[a-zA-Z]*CE(\d+)$', col)
        if m:
            deb_cols[int(m.group(1))] = col
        m2 = re.match(r'^FinCE(\d+)$', col)
        if m2:
            fin_cols[int(m2.group(1))] = col

    durations = []
    for n in sorted(deb_cols):
        if n in fin_cols:
            deb, fin = xlsx_row[deb_cols[n]], xlsx_row[fin_cols[n]]
            if pd.notna(deb) and pd.notna(fin):
                durations.append((fin - deb) * 86400)
    return durations


def extract_crises_from_annotations(ann, xlsx_row=None, start_marker='a', end_marker='b', peak_keyword='crise'):
    """
    Reconstruit les crises a partir du fichier d'annotations :
      1) toutes les paires (start_marker -> end_marker) trouvees (pairage glouton, gere
         plusieurs crises par fichier) donnent des intervalles "confirmes" (bornes fiables) ;
      2) toute annotation contenant peak_keyword (ex. 'crise', 'crise??') situee EN DEHORS
         d'un intervalle confirme est une crise "isolee" (bornee seulement par son instant
         d'apparition) ;
      3) si xlsx_row est fourni, la borne de fin manquante d'une crise isolee est completee
         en lui attribuant la duree exacte (FinCEn - DebCEn) du xlsx, appariee dans l'ordre
         chronologique des crises detectees.
    Retourne un DataFrame Crise / Debut (s) / Fin (s) / Duree (s) / Source.
    """
    idx_a = ann.index[ann['Annotations_clean'] == start_marker].tolist()
    idx_b = ann.index[ann['Annotations_clean'] == end_marker].tolist()

    # 1) pairage glouton des marqueurs de debut/fin
    pairs, used_b = [], set()
    for ia in sorted(idx_a):
        candidates = [ib for ib in idx_b if ib > ia and ib not in used_b]
        if candidates:
            ib = min(candidates)
            used_b.add(ib)
            pairs.append((ia, ib))

    confirmed = [{
        'Debut (s)': ann.loc[ia, 'Onset'], 'Fin (s)': ann.loc[ib, 'Onset'], 'Source': 'bornes a/b'
    } for ia, ib in pairs]

    # 2) marqueurs 'crise'-like hors des intervalles deja confirmes
    idx_crise_like = ann.index[ann['Annotations_clean'].str.contains(peak_keyword, na=False)].tolist()
    suspected = []
    for ic in idx_crise_like:
        t = ann.loc[ic, 'Onset']
        if not any(c['Debut (s)'] <= t <= c['Fin (s)'] for c in confirmed):
            suspected.append({
                'Debut (s)': t, 'Fin (s)': np.nan,
                'Source': f"'{ann.loc[ic, 'Annotations']}' isolee (sans bornes a/b)"
            })
        else:
            print(f"Info : marqueur '{ann.loc[ic, 'Annotations']}' (t={t:.2f}s) deja dans un intervalle a/b confirme, ignore.")

    events = sorted(confirmed + suspected, key=lambda c: c['Debut (s)'])

    # 3) completer les fins manquantes via les durees exactes du xlsx (appariees par ordre chronologique)
    if xlsx_row is not None:
        xlsx_durations = get_xlsx_crisis_durations(xlsx_row)
        for i, ev in enumerate(events):
            if pd.isna(ev['Fin (s)']):
                if i < len(xlsx_durations):
                    ev['Fin (s)'] = ev['Debut (s)'] + xlsx_durations[i]
                    ev['Source'] += f" + duree xlsx CE{i+1} ({xlsx_durations[i]:.1f}s)"
                else:
                    print(f"Attention : impossible de completer la fin de la crise a t={ev['Debut (s)']:.2f}s "
                          f"(pas assez de durees dans le xlsx) — a verifier/completer manuellement.")

    df_crises = pd.DataFrame(events)
    if len(df_crises) > 0:
        df_crises['Duree (s)'] = df_crises['Fin (s)'] - df_crises['Debut (s)']
        df_crises.index = range(1, len(df_crises) + 1)
        df_crises.index.name = 'Crise'
    return df_crises


ann = load_annotations(annotations_path)

# on charge la ligne xlsx du patient courant des maintenant, pour pouvoir completer
# les bornes de fin manquantes (voir cellule de verification croisee ci-dessous
# pour le detail du chargement / de la conversion d'identifiant)
CRISES_XLSX_PATH = f"{DATA_DIR}/Crises_absences.xlsx"

def patient_id_to_xlsx_id(patient_id):
    """Convertit '190304A_E_0000' -> '190304A-E' (identifiant utilise dans Crises_absences.xlsx)."""
    m = re.match(r'^(\d+[A-Z])_([A-Z])_\d+$', patient_id)
    return f"{m.group(1)}-{m.group(2)}" if m else patient_id

xlsx_row = None
try:
    crises_xlsx = pd.read_excel(CRISES_XLSX_PATH).dropna(how='all')
    xlsx_id = patient_id_to_xlsx_id(PATIENT_ID)
    match = crises_xlsx[crises_xlsx['Unnamed: 0'] == xlsx_id]
    if len(match) == 1:
        xlsx_row = match.iloc[0]
    else:
        print(f"Patient {xlsx_id} introuvable dans Crises_absences.xlsx (verifier l'identifiant).")
except FileNotFoundError:
    print(f"Fichier non trouve : {CRISES_XLSX_PATH}")

df_crises = extract_crises_from_annotations(ann, xlsx_row=xlsx_row)

print(f"\nNombre total de crises detectees : {len(df_crises)}")
df_crises

**Vérification croisée avec `Crises_absences.xlsx`** (nombre de crises attendu pour ce patient) :

In [ ]:
if xlsx_row is not None:
    nombre_ce_attendu = xlsx_row['NombreCE']
    print(f"Patient {patient_id_to_xlsx_id(PATIENT_ID)} : NombreCE (xlsx) = {nombre_ce_attendu}  |  "
          f"crises detectees (annotations) = {len(df_crises)}")
    if nombre_ce_attendu == len(df_crises):
        print("-> coherent.")
    else:
        print("-> ATTENTION, incoherent : verifier les marqueurs d'annotation pour ce patient "
              "(peut-etre un pattern different de 'a'/'crise'/'b').")

### Question 4

Statistiques sur les crises détectées : durée moyenne, minimale, maximale, et intervalle moyen entre deux crises.

In [ ]:
if len(df_crises) > 0:
    duree_moyenne = df_crises['Duree (s)'].mean()
    duree_min     = df_crises['Duree (s)'].min()
    duree_max     = df_crises['Duree (s)'].max()

    print(f"Duree moyenne des crises : {duree_moyenne:.2f} s")
    print(f"Duree minimale           : {duree_min:.2f} s")
    print(f"Duree maximale           : {duree_max:.2f} s")

    if len(df_crises) > 1:
        intervalles = df_crises['Debut (s)'].iloc[1:].values - df_crises['Fin (s)'].iloc[:-1].values
        intervalle_moyen = intervalles.mean()
        print(f"Intervalle moyen entre deux crises : {intervalle_moyen:.2f} s")
    else:
        intervalle_moyen = np.nan
        print("Une seule crise detectee : l'intervalle moyen entre deux crises n'est pas calculable.")
else:
    print("Aucune crise detectee pour ce patient.")

**Question :** Les crises ont-elles toutes une durée similaire ?

_(à compléter automatiquement une fois plusieurs crises disponibles — comparer `Duree (s).std()` à
la moyenne, ou tracer un histogramme des durées. Avec un seul patient à 1 crise, cette question
n'est réellement informative qu'en combinant les deux patients du binôme.)_

## Partie III — Construction de la variable cible

### Question 5

Création de la colonne `Class` : `1` pour les échantillons compris dans un intervalle de crise
(`Debut (s)` ≤ `Time` ≤ `Fin (s)`, d'après `df_crises`), `0` sinon.

In [1]:
def build_class_column(df, df_crises, time_col='Time'):
    """Retourne un array 0/1 : 1 si l'echantillon df[time_col] tombe dans un intervalle
    [Debut (s), Fin (s)] d'une crise de df_crises, 0 sinon."""
    classes = np.zeros(len(df), dtype=int)
    t = df[time_col].to_numpy()
    for _, c in df_crises.iterrows():
        mask = (t >= c['Debut (s)']) & (t <= c['Fin (s)'])
        classes[mask] = 1
    return classes


df['Class'] = build_class_column(df, df_crises)

n_total = len(df)
n_crise = int(df['Class'].sum())
n_hors  = n_total - n_crise

print(f"Nombre total d'echantillons : {n_total}")
print(f"Classe 0 (hors crise) : {n_hors} ({n_hors/n_total*100:.2f}%)")
print(f"Classe 1 (crise)      : {n_crise} ({n_crise/n_total*100:.2f}%)")
df[['Time', channels[0], 'Class']].iloc[[0, -1]]

NameError: name 'df' is not defined

### Question 6

Affichage du signal EEG en distinguant visuellement les périodes hors crise / pendant crise.

In [ ]:
channel_to_plot = channels[0]  # changer ici pour visualiser un autre canal

t   = df['Time'].to_numpy()
sig = df[channel_to_plot].to_numpy()
cls = df['Class'].to_numpy()

# on met a NaN les points de l'autre classe pour garder l'alignement temporel
# sans que matplotlib ne relie les segments entre eux par une ligne parasite
sig_hors  = np.where(cls == 0, sig, np.nan)
sig_crise = np.where(cls == 1, sig, np.nan)

plt.figure(figsize=(14, 4))
plt.plot(t, sig_hors, color='steelblue', linewidth=0.5, label='Hors crise')
plt.plot(t, sig_crise, color='red', linewidth=0.8, label='Crise')

# ombrage des intervalles de crise pour plus de lisibilite
for _, c in df_crises.iterrows():
    plt.axvspan(c['Debut (s)'], c['Fin (s)'], color='red', alpha=0.1)

plt.xlabel("Temps (s)")
plt.ylabel("Amplitude EEG (\u00b5V)")
plt.title(f"Signal EEG — canal {channel_to_plot} — patient {PATIENT_ID} (crises en rouge)")
plt.legend(loc='upper right')
plt.tight_layout()
plt.show()

**Question :** Le déséquilibre entre les deux classes est-il important ?

_(à compléter : ici, la classe 1 (crise) ne représente que ~1.3% des échantillons contre ~98.7%
pour la classe 0 — déséquilibre très fort, typique en détection d'événements rares. Cela aura des
implications importantes en Partie IX : l'Accuracy seule sera trompeuse, il faudra privilégier
Precision/Recall/F1 sur la classe 1.)_

## Partie IV — Segmentation du signal

### Question 7

Découpage du signal en fenêtres temporelles de **2 s** avec **50 % de recouvrement**
(à 256 Hz : fenêtre = 512 échantillons, pas = 256 échantillons = 1 s).

Chaque fenêtre est étiquetée `1` si **au moins 50 %** de ses échantillons sont en crise
(`Class == 1`), `0` sinon — on réutilise directement la colonne `Class` construite en Partie III,
ce qui garantit la cohérence entre l'étiquetage échantillon-par-échantillon et l'étiquetage
par fenêtre.

Le résultat est un nouveau dataset **une ligne = une fenêtre** (`df_windows`), utilisé pour
l'extraction des caractéristiques en Partie V.

In [ ]:
def segment_signal(df, fs, window_s=2.0, overlap=0.5, class_threshold=0.5):
    """
    Decoupe df (avec colonnes 'Time' et 'Class') en fenetres de duree window_s (s),
    avec un recouvrement 'overlap' (0-1). Chaque fenetre recoit :
      - Time_debut / Time_fin : bornes temporelles de la fenetre ;
      - Class : 1 si la proportion d'echantillons en crise dans la fenetre >= class_threshold.
    Retourne (df_windows, window_n, step_n) ou window_n/step_n sont exprimes en echantillons.
    """
    window_n = int(round(window_s * fs))
    step_n   = int(round(window_n * (1 - overlap)))

    time_vals  = df['Time'].to_numpy()
    class_vals = df['Class'].to_numpy()
    n_samples  = len(df)

    starts = range(0, n_samples - window_n + 1, step_n)

    rows = []
    for s in starts:
        e = s + window_n
        frac_crise = class_vals[s:e].mean()
        rows.append({
            'window_start_idx': s,
            'window_end_idx':   e,
            'Time_debut': time_vals[s],
            'Time_fin':   time_vals[e - 1],
            'Class': int(frac_crise >= class_threshold),
        })

    return pd.DataFrame(rows), window_n, step_n


WINDOW_S = 2.0   # taille de fenetre (s)
OVERLAP  = 0.5   # 50% de recouvrement

df_windows, window_n, step_n = segment_signal(df, fs, window_s=WINDOW_S, overlap=OVERLAP)

print(f"Taille de fenetre : {WINDOW_S} s = {window_n} echantillons")
print(f"Pas (step)        : {WINDOW_S * (1 - OVERLAP)} s = {step_n} echantillons ({OVERLAP*100:.0f}% de recouvrement)")
print(f"Nombre total de fenetres : {len(df_windows)}")
print(f"  Classe 0 (pas de crise) : {(df_windows['Class'] == 0).sum()} "
      f"({(df_windows['Class'] == 0).mean() * 100:.2f}%)")
print(f"  Classe 1 (crise)        : {(df_windows['Class'] == 1).sum()} "
      f"({(df_windows['Class'] == 1).mean() * 100:.2f}%)")

df_windows.head()